# 📚 Buscador de revistas Scopus + SCImago + IA gratuita (OpenRouter)
Ejecuta las celdas en orden. Necesitas tu API key de OpenRouter (gratis: https://openrouter.ai/keys), que se ingresa dentro de la app.

## 1. Obtener el código
Elige **A** (desde tu GitHub) o **B** (subir los archivos a mano).

In [ ]:
# A) Clonar desde GitHub (cambia la URL por la de tu repositorio)
REPO = 'https://github.com/TU_USUARIO/TU_REPO.git'
!rm -rf app_revistas && git clone $REPO app_revistas
%cd app_revistas

In [ ]:
# B) Alternativa: subir app.py, build_dataset.py, requirements.txt y data/revistas_procesadas.csv.gz
# (omite esta celda si usaste A)
import os
from google.colab import files
os.makedirs('app_revistas/data', exist_ok=True)
%cd app_revistas
up = files.upload()
for n in up:
    if n.endswith('.gz'): os.replace(n, 'data/'+n)

## 2. (Opcional) Generar la base desde tus archivos originales
Solo si no existe `data/revistas_procesadas.csv.gz`: sube el Excel de Scopus y el CSV de SCImago.

In [ ]:
import os
if not os.path.exists('data/revistas_procesadas.csv.gz'):
    from google.colab import files
    os.makedirs('data', exist_ok=True)
    print('Sube el .xlsx de Scopus y el .csv de SCImago')
    up = files.upload()
    x = [f for f in up if f.endswith('.xlsx')][0]
    c = [f for f in up if f.endswith('.csv')][0]
    !python build_dataset.py "$x" "$c"
else:
    print('Base ya disponible ✅')

## 3. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 4. Lanzar la app y abrir el enlace público

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
!pkill -f streamlit; pkill -f cloudflared
subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'], stdout=open('st.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)
tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8501'], stderr=subprocess.PIPE, text=True)
for line in tun.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('👉 Abre la app aquí:', m.group(0)); break

> Mantén esta celda en ejecución mientras uses la app (botón ■ para detenerla).